# Part 1 — Python & Data Foundations: ingest, validate, clean, enrich


In [1]:
import csv
import json

import numpy as np
import pandas as pd
import requests

DATA_DIR = "data/"

## Task 1.1 — Read the file the foundations way (csv module only)

In [2]:
def parse_row(row):
    """Return a copy of the row with age, fee_inr, prior_no_shows as int and distance_km as float."""
    parsed = dict(row)
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed


accepted, rejected = [], []
with open(DATA_DIR + "appointments.csv", newline="") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    rows = list(reader)

print("Rows read:", len(rows))

for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError as e:
        rejected.append({"row": row, "error": str(e)})

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))
print("  blank age:        ", sum(r["row"]["age"].strip() == "" for r in rejected))
print("  blank distance_km:", sum(r["row"]["distance_km"].strip() == "" for r in rejected))

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)
print("Saved rejected_rows.json")

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows read: 1020
Accepted: 988
Rejected: 32
  blank age:         12
  blank distance_km: 20
Saved rejected_rows.json


In [3]:
# A type check is not a validity check: impossible ages still pass parse_row
impossible = [r for r in accepted if r["age"] < 1 or r["age"] > 100]
print("Accepted rows with impossible ages:", len(impossible), [r["age"] for r in impossible])

Accepted rows with impossible ages: 5 [0, 150, -4, 250, 199]


## Task 1.2 — Inspect and clean, pass 1 (pandas)


In [4]:
df = pd.read_csv(DATA_DIR + "appointments.csv")
print("Shape:", df.shape)
df.info()

Shape: (1020, 14)
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB


In [5]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())

Missing values per column:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64

Exact duplicate rows: 20


In [6]:
# (a) drop exact duplicate rows
df = df.drop_duplicates()
print("After (a) dropping duplicates:", len(df))

# (b) clean booking_channel labels
print("\nbooking_channel raw spellings:", df["booking_channel"].nunique())
print(sorted(df["booking_channel"].unique()))
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("booking_channel after cleaning:", df["booking_channel"].nunique(), sorted(df["booking_channel"].unique()))
print("After (b):", len(df))

# (c) convert dates; unreadable appt_date values (e.g. 'TBD', '2025-02-30') become NaT
bad_dates = df.loc[pd.to_datetime(df["appt_date"], errors="coerce").isna(), "appt_date"]
print("\nappt_date values that cannot be read:", bad_dates.tolist())
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("After (c):", len(df))

# (d) drop rows with no usable appt_date or no status
print("\nMissing appt_date after conversion:", df["appt_date"].isna().sum())
print("Missing status:", df["status"].isna().sum())
df = df.dropna(subset=["appt_date", "status"])
print("After (d):", len(df))

After (a) dropping duplicates: 1000

booking_channel raw spellings: 16
[' App', ' phone', ' walk-in', 'APP ', 'App', 'App ', 'PHONE', 'Phone', 'Phone ', 'WALK-IN', 'Walk-In', 'Walk-in', 'Walk-in ', 'app', 'phone', 'walk-in']
booking_channel after cleaning: 3 ['App', 'Phone', 'Walk-In']
After (b): 1000

appt_date values that cannot be read: ['not booked', '2025-06-31', '2025-04-31', 'TBD', nan, '2025-02-30']
After (c): 1000

Missing appt_date after conversion: 6
Missing status: 4
After (d): 990


## Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [7]:
# (a) ages outside 1..100 are impossible -> missing
invalid_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages:", invalid_age.sum(), df.loc[invalid_age, "age"].tolist())
df.loc[invalid_age, "age"] = np.nan
print("Missing ages in total:", df["age"].isna().sum())

# (b) fill gaps with the median
age_median = df["age"].median()
dist_median = df["distance_km"].median()
print("Median age:", age_median, "| Median distance_km:", dist_median)
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)
print("Missing after fill -> age:", df["age"].isna().sum(), "distance_km:", df["distance_km"].isna().sum())

Invalid ages: 5 [0.0, 150.0, -4.0, 250.0, 199.0]
Missing ages in total: 17
Median age: 48.0 | Median distance_km: 8.25
Missing after fill -> age: 0 distance_km: 0


In [8]:
# (c) IQR capping for distance_km and fee_inr
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_high = (df[col] > upper).sum()
    n_low = (df[col] < lower).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(f"{col}: Q1={q1}, Q3={q3}, IQR={iqr}, lower fence={lower}, upper fence={upper}")
    print(f"   capped: {n_high + n_low} values ({n_high} high, {n_low} low)")

# (d) rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("\nLargest fee after capping and rounding:", df["fee_inr"].max())
print("Largest distance after capping and rounding:", df["distance_km"].max())

distance_km: Q1=4.625, Q3=13.0, IQR=8.375, lower fence=-7.9375, upper fence=25.5625
   capped: 26 values (26 high, 0 low)
fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, lower fence=68.75, upper fence=1218.75
   capped: 3 values (3 high, 0 low)

Largest fee after capping and rounding: 1219
Largest distance after capping and rounding: 25.6


## Task 1.4 — Features and a NumPy summary

In [9]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print(f"No-shows: {df['no_show'].sum()} of {len(df)} ({round(df['no_show'].mean() * 100, 1)}%)")

lead_days range: 1 to 45
No-shows: 218 of 990 (22.0%)


In [10]:
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) column means
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(X.mean(axis=0), 2))

# (b) boolean mask on lead_days
mask = X[:, 3] > 14
print(f"Rows with lead_days > 14: {mask.sum()} ({round(mask.mean() * 100, 1)}%)")

# (c) z-scores of distance_km (np.std = population std)
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("Rows with |z| > 2 for distance_km:", (np.abs(z) > 2).sum())

Column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
Rows with lead_days > 14: 296 (29.9%)
Rows with |z| > 2 for distance_km: 53


## Task 1.5 — Enrich the clinics from a live API (Open-Meteo).

In [11]:
clinics = pd.read_csv(DATA_DIR + "clinics.csv")
URL = "https://api.open-meteo.com/v1/forecast"


def fetch_temp(lat, lon):
    """Return (temp_c, reason). temp_c is None if the reply cannot be used."""
    try:
        r = requests.get(URL, params={"latitude": lat, "longitude": lon,
                                      "hourly": "temperature_2m", "forecast_days": 1}, timeout=10)
    except requests.exceptions.RequestException as e:
        return None, f"request failed: {e}"
    if r.status_code != 200:
        return None, f"status {r.status_code}"
    data = r.json()
    if "hourly" not in data or not data["hourly"].get("temperature_2m"):
        return None, "no hourly data in reply"
    return data["hourly"]["temperature_2m"][0], "ok"


temps = []
for _, c in clinics.iterrows():
    temp, reason = fetch_temp(c["latitude"], c["longitude"])
    if temp is None:                       # retry once
        temp, reason = fetch_temp(c["latitude"], c["longitude"])
    if temp is None:
        print(f"FAILED: {c['clinic_name']} -> {reason}; temp_c stored as None")
    else:
        print(f"{c['clinic_name']:<26} status 200, temp_c = {temp}")
    temps.append(temp)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
clinics

Kothrud Family Clinic      status 200, temp_c = 20.9


Hadapsar Care Centre       status 200, temp_c = 21.0


Sitabuldi Health Point     status 200, temp_c = 21.8


Malviya Nagar Clinic       status 200, temp_c = 19.8


Vaishali Nagar Clinic      status 200, temp_c = 19.8


Edappally Medical Centre   status 200, temp_c = 25.9


Vijay Nagar Clinic         status 200, temp_c = 20.5


Gomti Nagar Care Point     status 200, temp_c = 19.2


,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,20.9
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,21.0
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,21.8
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,19.8
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,19.8
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,25.9
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,20.5
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,19.2


## Task 1.6 — Export the clean table and write the SQL load script

In [12]:
COLUMNS = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
           "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent", "prior_no_shows",
           "status", "lead_days", "no_show"]

clean = df.sort_values("appointment_id")[COLUMNS].copy()
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")

print("appointment_id unique:", clean["appointment_id"].is_unique,
      f"({clean['appointment_id'].nunique()} distinct values)")
print("Missing values in the 16 columns:", clean.isnull().sum().sum())
assert clean["appointment_id"].is_unique and clean.isnull().sum().sum() == 0

clean.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv with shape", clean.shape)

appointment_id unique: True (990 distinct values)
Missing values in the 16 columns: 0
Saved appointments_clean.csv with shape (990, 16)


In [13]:
def sql_value(v):
    """Text in single quotes, numbers unquoted, missing values as NULL."""
    if pd.isna(v):
        return "NULL"
    if isinstance(v, str):
        return f"'{v}'"
    return f"{v}"


clinics_enriched = pd.read_csv("clinics_enriched.csv")
n_statements = 0
with open("load_data.sql", "w") as f:
    # clinics first: appointments.clinic_id is a foreign key to clinics
    for _, c in clinics_enriched.iterrows():
        values = ", ".join(sql_value(v) for v in c.tolist())
        f.write(f"INSERT INTO clinics VALUES ({values});\n")
        n_statements += 1
    for _, a in clean.iterrows():
        values = ", ".join(sql_value(v) for v in a.tolist())
        f.write(f"INSERT INTO appointments VALUES ({values});\n")
        n_statements += 1

print("INSERT statements written:", n_statements)
with open("load_data.sql") as f:
    lines = f.readlines()
print(lines[0], lines[8], lines[-1], sep="")

INSERT statements written: 998
INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 20.9);
INSERT INTO appointments VALUES ('A0001', 'P509', 'C05', '2025-01-24', '2025-01-25', 60, 'F', 18.0, 850, 'Phone', 'Consultation', 'Yes', 1, 'No-Show', 1, 1);
INSERT INTO appointments VALUES ('A1000', 'P085', 'C08', '2024-12-03', '2025-01-17', 38, 'M', 10.8, 550, 'App', 'Follow-up', 'Yes', 2, 'No-Show', 45, 1);

